In [4]:
import subprocess

smile_path = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe"
config_path = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\pitch.conf"
input_wav = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count\steth_20181001_11_01_50.wav"
output_csv = r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\pitch_test.csv"

cmd = [
    smile_path,
    "-C", config_path,
    "-I", input_wav,
    "-O", output_csv
]

subprocess.run(cmd, check=True)
print("Done!")

CalledProcessError: Command '['E:\\Research Project (Prof. Preeti Rao)\\opensmile-3.0-win-x64\\bin\\SMILExtract.exe', '-C', 'E:\\Research Project (Prof. Preeti Rao)\\opensmile-3.0-win-x64\\myconfig\\pitch.conf', '-I', 'E:\\Research Project (Prof. Preeti Rao)\\Top 100 files by Wheeze count\\steth_20181001_11_01_50.wav', '-O', 'E:\\Research Project (Prof. Preeti Rao)\\Classification_24B3907\\pitch_test.csv']' returned non-zero exit status 4294967295.

In [6]:
import pandas as pd

# Load your CSV
df = pd.read_csv(r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\energy_test.csv", header=None)

# Split the single column into three columns
df_split = df[0].str.split(";", expand=True)

# Optional: rename columns
df_split.columns = ["frameIndex", "frameTime", "pcm_LOGenergy"]

# Save to new CSV
df_split.to_csv("cleaned_output.csv", index=False)

print(df_split.head())

   frameIndex  frameTime  pcm_LOGenergy
0  frameIndex  frameTime  pcm_LOGenergy
1           0   0.000000  -7.873582e+00
2           1   0.010000  -8.058037e+00
3           2   0.020000  -8.130335e+00
4           3   0.030000  -7.946619e+00


In [7]:
import pandas as pd

# 1. Load the already cleaned CSV 
df = pd.read_csv("cleaned_output.csv")  # Uses header automatically

# Convert frameTime to numeric (handles header & mixed types)
df["frameTime"] = pd.to_numeric(df["frameTime"], errors='coerce')

# 2. Load wheeze label file (tab-separated, NO header)
label_df = pd.read_csv(
    r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count\steth_20181001_11_01_50_label_audacity.txt",
    sep="\t",
    header=None,  # CRITICAL: Audacity export has NO header row
    names=["start", "end", "label"]
)

# Only keep Wheeze intervals
wheeze_intervals = label_df[label_df["label"] == "Wheeze"][["start", "end"]].values.tolist()

# 3. Vectorized Wheeze labeling (fast)
def is_wheeze_vectorized(frame_times, intervals):
    labels = pd.Series(0, index=frame_times.index)
    for start, end in intervals:
        mask = (frame_times >= start) & (frame_times <= end)
        labels[mask] = 1
    return labels

df["Wheeze"] = is_wheeze_vectorized(df["frameTime"], wheeze_intervals)

df = df[(df["frameTime"].notna()) & (pd.to_numeric(df["frameIndex"], errors='coerce').notna())].reset_index(drop=True)

df.to_csv("final_with_wheeze_labels.csv", index=False)

print("Wheeze column added successfully!")
print(df[["frameTime", "Wheeze"]].head(10))
print(f"Wheeze frames: {df['Wheeze'].sum()}")

Wheeze column added successfully!
   frameTime  Wheeze
0       0.00       0
1       0.01       0
2       0.02       0
3       0.03       0
4       0.04       0
5       0.05       0
6       0.06       0
7       0.07       0
8       0.08       0
9       0.09       0
Wheeze frames: 667


In [10]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import numpy as np

# Load the cleaned CSV
df = pd.read_csv("final_with_wheeze_labels.csv")  # Use your cleaned version

# Verify data quality
print("Dataset shape:", df.shape)
print("Wheeze distribution:\n", df['Wheeze'].value_counts())

# Feature engineering for audio frames
df['log_energy'] = np.log1p(np.abs(df['pcm_LOGenergy']))  # Log energy feature
df['energy_norm'] = (df['pcm_LOGenergy'] - df['pcm_LOGenergy'].mean()) / df['pcm_LOGenergy'].std()  # Normalized energy
df['time_norm'] = df['frameTime'] / df['frameTime'].max()  # Normalized time position[file:1]
df.to_csv("energy_features.csv", index=False)

# Features and target
#features = ['pcm_LOGenergy', 'log_energy', 'energy_norm', 'time_norm']
features = ['log_energy', 'energy_norm', 'time_norm']
X = df[features]
y = df['Wheeze']

# Train-test split (stratified for imbalanced wheeze data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# XGBoost classifier optimized for imbalanced audio classification
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),  # Handle class imbalance
    random_state=42,
    eval_metric='auc'
)

# Train model
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=10
)

# Predictions and evaluation
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))
print(f"\nAUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

model.save_model("xgb_sentiment_model.json")
print("Model saved!")

Dataset shape: (1498, 4)
Wheeze distribution:
 Wheeze
0    831
1    667
Name: count, dtype: int64
[0]	validation_0-auc:0.94673	validation_1-auc:0.91978
[10]	validation_0-auc:0.96620	validation_1-auc:0.93513
[20]	validation_0-auc:0.99005	validation_1-auc:0.97512
[30]	validation_0-auc:0.99741	validation_1-auc:0.98858
[40]	validation_0-auc:0.99881	validation_1-auc:0.99083
[50]	validation_0-auc:0.99939	validation_1-auc:0.99470
[60]	validation_0-auc:0.99962	validation_1-auc:0.99488
[70]	validation_0-auc:0.99976	validation_1-auc:0.99479
[80]	validation_0-auc:0.99983	validation_1-auc:0.99496
[90]	validation_0-auc:0.99988	validation_1-auc:0.99541
[100]	validation_0-auc:0.99991	validation_1-auc:0.99546
[110]	validation_0-auc:0.99990	validation_1-auc:0.99541
[120]	validation_0-auc:0.99990	validation_1-auc:0.99492
[130]	validation_0-auc:0.99992	validation_1-auc:0.99514
[140]	validation_0-auc:0.99995	validation_1-auc:0.99564
[150]	validation_0-auc:0.99997	validation_1-auc:0.99546
[160]	validation_